In [16]:
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

student_data = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]

course_data = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('CREATE TABLE student (student_id INT, name TEXT, class TEXT, course_id INT, score REAL)')
cursor.executemany('INSERT INTO student VALUES (?,?,?,?,?)', student_data)

cursor.execute('CREATE TABLE course (id INT, course_name TEXT)')
cursor.executemany('INSERT INTO course VALUES (?,?)', course_data)

def query(sql):
    return pd.read_sql_query(sql, conn)

# Tích Decartes.

In [17]:
query("SELECT * FROM student CROSS JOIN course")

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


#  INNER JOIN

In [18]:
sql_inner = """
SELECT s.name, s.course_id, c.course_name
FROM student s
INNER JOIN course c ON s.course_id = c.id
"""
query(sql_inner)

,name,course_id,course_name
0,Nguyen Minh Hoang,12,Giai tich
1,Tran Thi Lan,34,Thong ke
2,Bui Tien Dung,34,Thong ke


#  LEFT JOIN

In [19]:
sql_left = """
SELECT s.name, s.course_id, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
"""
query(sql_left)

,name,course_id,course_name
0,Nguyen Minh Hoang,12.0,Giai tich
1,Tran Thi Lan,34.0,Thong ke
2,Pham Van Nam,NaN,None
3,Le Thanh Huyen,20.0,None
4,Vu Quoc Anh,24.0,None
5,Dang Thuy Linh,24.0,None
6,Bui Tien Dung,34.0,Thong ke
7,Ho Ngoc Mai,20.0,None
8,Duong Huu Phuc,NaN,None
9,Cao Thi Hanh,NaN,None


# RIGHT JOIN

In [20]:
sql_right = """
SELECT s.name, c.id, c.course_name
FROM course c
LEFT JOIN student s ON c.id = s.course_id
"""
query(sql_right)

,name,id,course_name
0,Nguyen Minh Hoang,12,Giai tich
1,Bui Tien Dung,34,Thong ke
2,Tran Thi Lan,34,Thong ke
3,None,26,Tin hoc


# FULL OUTER JOIN

In [21]:
sql_full_outer = """
SELECT s.name, s.course_id, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT s.name, c.id, c.course_name
FROM course c
LEFT JOIN student s ON c.id = s.course_id
"""
query(sql_full_outer)

,name,course_id,course_name
0,None,26.0,Tin hoc
1,Bui Tien Dung,34.0,Thong ke
2,Cao Thi Hanh,NaN,None
3,Dang Thuy Linh,24.0,None
4,Duong Huu Phuc,NaN,None
5,Ho Ngoc Mai,20.0,None
6,Le Thanh Huyen,20.0,None
7,Nguyen Minh Hoang,12.0,Giai tich
8,Pham Van Nam,NaN,None
9,Tran Thi Lan,34.0,Thong ke


# Cập nhật giá trị course_id thiếu và loại bỏ bản ghi không tồn tại

In [22]:
cursor.execute("UPDATE student SET course_id = 26 WHERE course_id IS NULL")

cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course)")

## a. Tổng số sinh viên, điểm trung bình của từng lớp.

In [23]:
sql_a = """
SELECT 
    class AS 'Lớp', 
    COUNT(*) AS 'Tổng số sinh viên', 
    ROUND(AVG(score), 2) AS 'Điểm trung bình'
FROM student
GROUP BY class
"""
print("--- Kết quả câu a ---")
display(query(sql_a))

--- Kết quả câu a ---


,Lớp,Tổng số sinh viên,Điểm trung bình
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


## b. Tổng số sinh viên, điểm trung bình của từng môn học.

In [24]:
sql_b = """
SELECT 
    c.course_name AS 'Tên môn học',
    COUNT(s.student_id) AS 'Tổng số sinh viên',
    ROUND(AVG(s.score), 2) AS 'Điểm trung bình'
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
print("--- Kết quả câu b: Thống kê môn học ---")
display(query(sql_b))

--- Kết quả câu b: Thống kê môn học ---


,Tên môn học,Tổng số sinh viên,Điểm trung bình
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


## c. Phân loại thi đua theo số điểm của từng môn học biết: 
- i. Điểm TB ≥  9.0: Xuất sắc. 
- ii. 6.0 ≤ Điểm TB ≤ 8.9: Tốt. 
- iii. Điểm TB < 6.0: Kém.

In [25]:
sql_c = """
SELECT 
    c.course_name AS 'Tên môn học',
    CASE 
        WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
        WHEN AVG(s.score) >= 6.0 THEN 'Tốt'
        ELSE 'Kém'
    END AS 'Phân loại thi đua'
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
print("--- Kết quả câu c: Phân loại thi đua ---")
display(query(sql_c))

--- Kết quả câu c: Phân loại thi đua ---


,Tên môn học,Phân loại thi đua
0,Giai tich,Tốt
1,Thong ke,Xuất sắc
2,Tin hoc,Tốt


In [26]:
rank_sql = """
SELECT name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) as global_rank,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) as class_rank,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) as course_rank
FROM student
"""
df_rank = query(rank_sql)


print("TOP 3 sinh viên đạt hạng cao nhất ---")
top_3_global = df_rank.nsmallest(3, 'global_rank', keep='all')
print(top_3_global[['name', 'score', 'global_rank']])

print("\nTOP 3 sinh viên đạt hạng thấp nhất")
bottom_3_global = df_rank.nlargest(3, 'global_rank', keep='all')
print(bottom_3_global[['name', 'score', 'global_rank']])


print("\n TOP 3 sinh viên đạt hạng cao nhất từng lớp")
top_3_class = df_rank[df_rank['class_rank'] <= 3].sort_values(['class', 'class_rank'])
print(top_3_class[['class', 'name', 'score', 'class_rank']])

print("\n TOP 3 sinh viên đạt hạng thấp nhất từng lớp")
bottom_3_class = df_rank.groupby('class').apply(lambda x: x.nlargest(3, 'class_rank')).reset_index(drop=True)
print(bottom_3_class[['class', 'name', 'score', 'class_rank']])


print("\n TOP 3 sinh viên đạt hạng cao nhất theo môn học")
top_3_course = df_rank[df_rank['course_rank'] <= 3].sort_values(['course_id', 'course_rank'])
print(top_3_course[['course_id', 'name', 'score', 'course_rank']])

print("\n TOP 3 sinh viên đạt hạng thấp nhất theo môn học")
bottom_3_course = df_rank.groupby('course_id').apply(lambda x: x.nlargest(3, 'course_rank')).reset_index(drop=True)
print(bottom_3_course[['course_id', 'name', 'score', 'course_rank']])

TOP 3 sinh viên đạt hạng cao nhất ---
            name  score  global_rank
0   Tran Thi Lan    9.2            1
1  Bui Tien Dung    9.2            1
2   Pham Van Nam    7.9            3

TOP 3 sinh viên đạt hạng thấp nhất
                name  score  global_rank
5  Nguyen Minh Hoang    6.7            6
4       Cao Thi Hanh    7.0            5
3     Duong Huu Phuc    7.2            4

 TOP 3 sinh viên đạt hạng cao nhất từng lớp
      class               name  score  class_rank
0   Kinh Te       Tran Thi Lan    9.2           1
1   Kinh Te      Bui Tien Dung    9.2           1
3   Kinh Te     Duong Huu Phuc    7.2           3
4  May Tinh       Cao Thi Hanh    7.0           1
5  May Tinh  Nguyen Minh Hoang    6.7           2
2  Toan Tin       Pham Van Nam    7.9           1

 TOP 3 sinh viên đạt hạng thấp nhất từng lớp
      class               name  score  class_rank
0   Kinh Te     Duong Huu Phuc    7.2           3
1   Kinh Te       Tran Thi Lan    9.2           1
2   Kinh Te      Bui Ti

C:\Users\Admin\AppData\Local\Temp\ipykernel_18304\1401541026.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bottom_3_class = df_rank.groupby('class').apply(lambda x: x.nlargest(3, 'class_rank')).reset_index(drop=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_18304\1401541026.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bottom_3_course = df_rank.groupby('course_id').apply(lambda x: x.nlarges

# Thêm trường graduation_date

In [27]:
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
except:
    pass 

update_grad_sql = """
WITH Ranked AS (
    SELECT student_id, RANK() OVER (ORDER BY score DESC) as rnk FROM student
)
UPDATE student 
SET graduation_date = datetime('now', '+' || (SELECT rnk FROM Ranked WHERE Ranked.student_id = student.student_id) || ' days')
"""
cursor.execute(update_grad_sql)
query("SELECT name, score, graduation_date FROM student")

,name,score,graduation_date
0,Nguyen Minh Hoang,6.7,2026-04-04 09:21:45
1,Tran Thi Lan,9.2,2026-03-30 09:21:45
2,Pham Van Nam,7.9,2026-04-01 09:21:45
3,Bui Tien Dung,9.2,2026-03-30 09:21:45
4,Duong Huu Phuc,7.2,2026-04-02 09:21:45
5,Cao Thi Hanh,7.0,2026-04-03 09:21:45
